## Sweep Analysis

Visualises the results of parameter sweeps produced by `experiments/sweep.py`.

**Workflow:**
1. Run a sweep from the terminal, e.g.  
   `python -m experiments.sweep --param dropout --values 0 0.05 0.10 0.20 0.40 --no-myopic`
2. Come back here, run the *Setup* and *Load data* cells, then run the plot sections you need.

Each plot section is self-contained — skip sections whose sweep you haven't run yet.

### Setup

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

repo_root = Path(r"C:\Users\49160\Adaptive-Onboarding")
sys.path.append(str(repo_root))

from src.plots import (
    ACTIVE_BLACK, PALETTE, POSTERIOR_GREEN, PRIOR_BLUE,
    QUESTION_ORANGE, STRUCTURE_GRAY,
    apply_notebook_style, save, style_ax,
)

apply_notebook_style()

RESULTS_DIR = repo_root / "experiments" / "results"

# Consistent style per policy — used in every plot
POLICY_STYLE: dict[str, dict] = {
    "fixed":               {"color": STRUCTURE_GRAY,  "ls": "--", "marker": "s", "label": "Fixed"},
    "random":              {"color": PRIOR_BLUE,       "ls": "--", "marker": "^", "label": "Random"},
    "myopic_exact":        {"color": ACTIVE_BLACK,     "ls": "-",  "marker": "D", "label": "Myopic (exact)"},
    "surrogate_unweighted":{"color": QUESTION_ORANGE,  "ls": "-",  "marker": "o", "label": "Surrogate (unweighted)"},
    "surrogate_weighted":  {"color": POSTERIOR_GREEN,  "ls": "-",  "marker": "o", "label": "Surrogate (weighted)"},
}

print(f"Results directory: {RESULTS_DIR}")
print(f"Available files : {[f.name for f in sorted(RESULTS_DIR.glob('*.json'))]}")

### Load data

In [ ]:
def load_sweep(param: str) -> dict:
    """
    Load the most recent sweep JSON for the given parameter name.
    E.g. load_sweep("dropout") finds the latest *sweep_dropout*.json.
    """
    pattern = f"*sweep_{param.replace('-', '_')}*.json"
    files = sorted(RESULTS_DIR.glob(pattern))
    if not files:
        raise FileNotFoundError(
            f"No sweep file found for param='{param}' in {RESULTS_DIR}.\n"
            f"Run: python -m experiments.sweep --param {param} --values ..."
        )
    path = files[-1]   # most recent
    print(f"Loading: {path.name}")
    return json.loads(path.read_text(encoding="utf-8"))


def load_run(pattern: str = "*.json") -> dict:
    """
    Load the most recent single-run JSON (not a sweep).
    Useful for sanity-checking a single condition.
    """
    files = [f for f in sorted(RESULTS_DIR.glob(pattern)) if "sweep" not in f.name]
    if not files:
        raise FileNotFoundError(f"No single-run JSON found matching '{pattern}'.")
    path = files[-1]
    print(f"Loading: {path.name}")
    return json.loads(path.read_text(encoding="utf-8"))


def sweep_series(data: dict, metric: str) -> dict[str, tuple[list, list]]:
    """
    Extract (sweep_values, metric_values) per policy from a sweep JSON.
    Returns {policy_name: (x_values, y_values)}.
    """
    series: dict[str, tuple[list, list]] = {}
    for condition in data["conditions"]:
        x = condition["value"]
        for policy, metrics in condition["policies"].items():
            if policy not in series:
                series[policy] = ([], [])
            series[policy][0].append(x)
            series[policy][1].append(metrics[metric])
    return series


print("Helpers ready.")

---
## 1 — Dropout probability sweep

**Run first:**
```bash
python -m experiments.sweep --param dropout --values 0 0.05 0.10 0.20 0.40 --no-myopic
# Add myopic for the final thesis run (slow):
python -m experiments.sweep --param dropout --values 0 0.05 0.10 0.20 0.40
```

In [ ]:
dropout_sweep = load_sweep("dropout")

In [ ]:
# ── Figure 1a: Dropout rate vs dropout probability ────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))

series = sweep_series(dropout_sweep, "dropout_rate")
for policy, (xs, ys) in series.items():
    s = POLICY_STYLE[policy]
    ax.plot(np.array(xs) * 100, np.array(ys) * 100,
            color=s["color"], ls=s["ls"], marker=s["marker"],
            markersize=5, linewidth=1.6, label=s["label"])

ax.set_xlabel("Per-question dropout probability on sensitive items (%)")
ax.set_ylabel("Episode dropout rate (%)")
ax.set_title("Dropout rate vs dropout pressure")
style_ax(ax, grid_axis="y")
ax.legend()
plt.tight_layout()
# save(fig, "fig_dropout_rate_vs_pressure")  # uncomment to export
plt.show()

In [ ]:
# ── Figure 1b: D-error vs dropout probability ─────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))

series = sweep_series(dropout_sweep, "mean_final_d_error")
for policy, (xs, ys) in series.items():
    s = POLICY_STYLE[policy]
    ax.plot(np.array(xs) * 100, ys,
            color=s["color"], ls=s["ls"], marker=s["marker"],
            markersize=5, linewidth=1.6, label=s["label"])

ax.set_xlabel("Per-question dropout probability on sensitive items (%)")
ax.set_ylabel(r"Mean D-error  $\det(\Sigma_T)^{1/d}$")
ax.set_title("Residual uncertainty vs dropout pressure")
style_ax(ax, grid_axis="y")
ax.legend()
plt.tight_layout()
# save(fig, "fig_d_error_vs_dropout")  # uncomment to export
plt.show()

In [ ]:
# ── Figure 1c: Sensitive questions asked vs dropout probability ───────────
fig, ax = plt.subplots(figsize=(7, 4))

series = sweep_series(dropout_sweep, "mean_sensitive_asked")
for policy, (xs, ys) in series.items():
    s = POLICY_STYLE[policy]
    ax.plot(np.array(xs) * 100, ys,
            color=s["color"], ls=s["ls"], marker=s["marker"],
            markersize=5, linewidth=1.6, label=s["label"])

ax.set_xlabel("Per-question dropout probability on sensitive items (%)")
ax.set_ylabel("Mean sensitive questions presented")
ax.set_title("Sensitive item exposure vs dropout pressure")
style_ax(ax, grid_axis="y")
ax.legend()
plt.tight_layout()
# save(fig, "fig_sensitive_asked_vs_dropout")  # uncomment to export
plt.show()

---
## 2 — Sensitive fraction sweep

**Run first:**
```bash
python -m experiments.sweep --param sensitive-frac --values 0.1 0.2 0.3 0.4 0.5 --no-myopic
```

In [ ]:
sfrac_sweep = load_sweep("sensitive_frac")

In [ ]:
# ── Figure 2a: Dropout rate vs sensitive fraction ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric, ylabel, title in [
    (axes[0], "dropout_rate",       "Episode dropout rate (%)",              "Dropout rate vs sensitive fraction"),
    (axes[1], "mean_final_d_error", r"Mean D-error  $\det(\Sigma_T)^{1/d}$", "D-error vs sensitive fraction"),
]:
    series = sweep_series(sfrac_sweep, metric)
    for policy, (xs, ys) in series.items():
        s = POLICY_STYLE[policy]
        yvals = np.array(ys) * 100 if metric == "dropout_rate" else ys
        ax.plot(np.array(xs) * 100, yvals,
                color=s["color"], ls=s["ls"], marker=s["marker"],
                markersize=5, linewidth=1.6, label=s["label"])
    ax.set_xlabel("Fraction of sensitive items in bank (%)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    style_ax(ax, grid_axis="y")
    ax.legend(fontsize=8)

plt.tight_layout()
# save(fig, "fig_sensitive_frac_sweep")  # uncomment to export
plt.show()

---
## 3 — Horizon sweep

**Run first:**
```bash
# 2D
python -m experiments.sweep --param horizon --values 4 8 12 16 20 --dropout 0.10 --no-myopic
# 6D
python -m experiments.sweep --param horizon --values 6 12 18 24 36 --dim 6 --dropout 0.10 --no-myopic
```

If you ran both, load whichever is most relevant below.

In [ ]:
horizon_sweep = load_sweep("horizon")

In [ ]:
# ── Figure 3: D-error and estimation error vs horizon ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

dim = horizon_sweep["fixed_config"].get("dim", "?")

for ax, metric, ylabel, title in [
    (axes[0], "mean_final_d_error",     r"Mean D-error  $\det(\Sigma_T)^{1/d}$",
     f"Residual uncertainty vs horizon  (dim={dim})"),
    (axes[1], "mean_estimation_error",  r"Mean estimation error  $\|\mu_T - \theta\|_2$",
     f"Estimation error vs horizon  (dim={dim})"),
]:
    series = sweep_series(horizon_sweep, metric)
    for policy, (xs, ys) in series.items():
        s = POLICY_STYLE[policy]
        ax.plot(xs, ys,
                color=s["color"], ls=s["ls"], marker=s["marker"],
                markersize=5, linewidth=1.6, label=s["label"])
    ax.set_xlabel("Horizon (questions per episode)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    style_ax(ax, grid_axis="y")
    ax.legend(fontsize=8)

plt.tight_layout()
# save(fig, "fig_horizon_sweep")  # uncomment to export
plt.show()

---
## 4 — Dimension sweep

**Run first:**
```bash
python -m experiments.sweep --param dim --values 2 4 6 8 --horizon 20 --dropout 0.10 --items 50 --no-myopic
```

In [ ]:
dim_sweep = load_sweep("dim")

In [ ]:
# ── Figure 4: Key metrics vs latent dimension ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, metric, ylabel, title in [
    (axes[0], "dropout_rate",         "Episode dropout rate (%)",              "Dropout rate vs dimension"),
    (axes[1], "mean_final_d_error",   r"Mean D-error  $\det(\Sigma_T)^{1/d}$", "D-error vs dimension"),
    (axes[2], "mean_estimation_error",r"Mean estimation error",                 "Estimation error vs dimension"),
]:
    series = sweep_series(dim_sweep, metric)
    for policy, (xs, ys) in series.items():
        s = POLICY_STYLE[policy]
        yvals = np.array(ys) * 100 if metric == "dropout_rate" else ys
        ax.plot(xs, yvals,
                color=s["color"], ls=s["ls"], marker=s["marker"],
                markersize=5, linewidth=1.6, label=s["label"])
    ax.set_xlabel("Latent dimensions")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    style_ax(ax, grid_axis="y")
    ax.legend(fontsize=8)

plt.tight_layout()
# save(fig, "fig_dimension_sweep")  # uncomment to export
plt.show()

---
## 5 — Single run summary

Quick bar chart of all metrics for a single run — useful for a first look at any new configuration.

In [ ]:
run = load_run()

In [ ]:
# ── Figure 5: Bar chart summary for one run ────────────────────────────────
policies  = list(run["policies"].keys())
colors    = [POLICY_STYLE[p]["color"] for p in policies]
labels    = [POLICY_STYLE[p]["label"] for p in policies]
cfg       = run["config"]

metrics_to_plot = [
    ("dropout_rate",         "Dropout rate",       True,  "%"),
    ("mean_final_d_error",   "D-error",             False, ""),
    ("mean_estimation_error","Est. error",          False, ""),
    ("mean_sensitive_asked", "Sens. asked",         False, ""),
]

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(14, 4))
fig.suptitle(
    f"Single run  —  dim={cfg['dim']}, horizon={cfg['horizon']}, "
    f"dropout={cfg['p_dropout_sens']*100:.0f}%",
    fontsize=10,
)

x = np.arange(len(policies))
for ax, (metric, title, as_pct, unit) in zip(axes, metrics_to_plot):
    vals = [run["policies"][p][metric] for p in policies]
    if as_pct:
        vals = [v * 100 for v in vals]
    bars = ax.bar(x, vals, color=colors, width=0.6, edgecolor="white", linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
    ax.set_title(title + (f" ({unit})" if unit else ""))
    style_ax(ax, grid_axis="y")

plt.tight_layout()
# save(fig, "fig_single_run_summary")  # uncomment to export
plt.show()

---
## Notes

_Record observations from the sweeps here._

### Dropout sweep

- 

### Sensitive fraction sweep

- 

### Horizon sweep

- 

### Dimension sweep

- 